# GrayMatter Ablation Study

Train and evaluate the three skip-connection variants of the Hybrid Attention U-Net:

| Variant | `skip_mode` | Description |
|---------|-------------|-------------|
| Plain U-Net | `identity` | Baseline — no skip attention |
| Coordinate Attention | `coord_only` | Triaxial coordinate gating |
| Full CISA | `full` | Coordinate gating + inter-slice convolution |

All variants share the same 4-level 3D U-Net backbone (channels `[32, 64, 128, 256]`), 
preprocessing, training recipe, and data splits.

In [ ]:
import importlib.util, subprocess, sys
for pkg in ["monai", "nibabel", "tqdm"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

## 1. Configuration

In [ ]:
import json, os, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ── Select variant ──────────────────────────────────────────────────
VARIANT    = "coord_attention"   # "plain_unet" | "coord_attention" | "full_cisa"
SKIP_MODES = {"plain_unet": "identity", "coord_attention": "coord_only", "full_cisa": "full"}
SKIP_MODE  = SKIP_MODES[VARIANT]
# ───────────────────────────────────────────────────────────────────

FOLDS       = [1, 2, 3, 4, 5]
MAX_EPOCHS  = 300
EARLY_STOP  = 30
BATCH_SIZE  = 2
LR          = 5e-4
ROI         = (48, 64, 48)
SEED        = 42

def _resolve_dataset() -> Path:
    for p in [Path("dataset"), Path("../dataset"), *Path("/kaggle/input").glob("graymatter-dataset*")]:
        if p.is_dir() and (p / "manifests").is_dir():
            return p
    raise FileNotFoundError("Dataset not found. Place dataset/ next to the notebook.")

DATASET_DIR = _resolve_dataset()
OUTPUT_DIR  = Path("ai/results/ablations") / VARIANT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Variant: {VARIANT} (skip_mode={SKIP_MODE})")
print(f"Device:  {device}")

## 2. Model Architecture

In [ ]:
from typing import Literal, Sequence
import torch.nn as nn

SkipMode = Literal["identity", "coord_only", "full"]

def _gn(ch):
    g = min(8, ch)
    while g > 1 and ch % g != 0: g -= 1
    return nn.GroupNorm(g, ch)

class DoubleConv3D(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv3d(i, o, 3, padding=1, bias=False), _gn(o), nn.ReLU(True),
            nn.Conv3d(o, o, 3, padding=1, bias=False), _gn(o), nn.ReLU(True),
        )
    def forward(self, x): return self.b(x)

class CISA(nn.Module):
    """Coordinate Inter-Slice Attention for skip connections."""
    def __init__(self, ch: int, mode: SkipMode = "coord_only"):
        super().__init__()
        self.mode = mode
        if mode == "identity": return
        mid = max(ch // 8, 8)
        def br():
            return nn.Sequential(nn.Conv3d(ch, mid, 1, bias=False), nn.ReLU(True), nn.Conv3d(mid, ch, 1, bias=False))
        self.bd, self.bh, self.bw = br(), br(), br()
        self.sig = nn.Sigmoid()
        if mode == "full":
            self.inter = nn.Sequential(
                nn.Conv3d(ch, ch, (3,1,1), padding=(1,0,0), groups=ch, bias=False),
                _gn(ch), nn.ReLU(True),
            )
        else:
            self.inter = None

    def forward(self, x):
        if self.mode == "identity": return x
        g = x * self.sig(self.bd(x.mean(2, keepdim=True)))
        g = g * self.sig(self.bh(g.mean(3, keepdim=True)))
        g = g * self.sig(self.bw(g.mean(4, keepdim=True)))
        if self.mode == "coord_only": return g
        return g + self.inter(g)

class HybridAttentionUNet3D(nn.Module):
    def __init__(self, skip_mode: SkipMode = "coord_only"):
        super().__init__()
        c = [32, 64, 128, 256]
        self.e1 = DoubleConv3D(1, c[0])
        self.e2 = DoubleConv3D(c[0], c[1])
        self.e3 = DoubleConv3D(c[1], c[2])
        self.e4 = DoubleConv3D(c[2], c[3])
        self.pool = nn.MaxPool3d(2)
        self.bn = nn.Sequential(DoubleConv3D(c[3], c[3]), nn.Dropout3d(0.1))
        self.s4 = CISA(c[3], mode=skip_mode)
        self.s3 = CISA(c[2], mode=skip_mode)
        self.s2 = CISA(c[1], mode=skip_mode)
        self.s1 = CISA(c[0], mode=skip_mode)
        self.u4 = nn.ConvTranspose3d(c[3], c[3], 2, 2)
        self.u3 = nn.ConvTranspose3d(c[3], c[2], 2, 2)
        self.u2 = nn.ConvTranspose3d(c[2], c[1], 2, 2)
        self.u1 = nn.ConvTranspose3d(c[1], c[0], 2, 2)
        self.d4 = DoubleConv3D(c[3]*2, c[3])
        self.d3 = DoubleConv3D(c[2]*2, c[2])
        self.d2 = DoubleConv3D(c[1]*2, c[1])
        self.d1 = DoubleConv3D(c[0]*2, c[0])
        self.out = nn.Conv3d(c[0], 3, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2)); e4 = self.e4(self.pool(e3))
        b = self.bn(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), self.s4(e4)], 1))
        d3 = self.d3(torch.cat([self.u3(d4), self.s3(e3)], 1))
        d2 = self.d2(torch.cat([self.u2(d3), self.s2(e2)], 1))
        d1 = self.d1(torch.cat([self.u1(d2), self.s1(e1)], 1))
        return self.out(d1)

print(f"Model: HybridAttentionUNet3D (skip_mode={SKIP_MODE})")

## 3. Data Transforms & Loaders

In [ ]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, ScaleIntensityRanged,
    SpatialPadd, ResizeWithPadOrCropd, EnsureTyped, RandFlipd, AsDiscreted,
)
from monai.data import CacheDataset, DataLoader, list_data_collate

def _find(base, name):
    p = base / name
    if p.exists(): return p
    if name.endswith(".nii.gz"):
        alt = base / name.replace(".nii.gz", ".nii")
        if alt.exists(): return alt
    return p

def _load_cases(manifest, split):
    cases = manifest["training"]["cases"] if split == "train" else manifest["validation"]["cases"]
    out = []
    for c in cases:
        for key in ("image", "label"):
            rel = Path(c[key]).relative_to("dataset")
            c[key] = str(_find(DATASET_DIR / rel.parent, rel.name))
        out.append(c)
    return out

def train_tf():
    return Compose([
        LoadImaged(["image", "label"], image_only=False),
        EnsureChannelFirstd(["image", "label"]),
        Orientationd(["image", "label"], axcodes="RAS"),
        ScaleIntensityRanged("image", a_min=0, a_max=2500, b_min=0, b_max=1, clip=True),
        AsDiscreted("label"),
        SpatialPadd(["image", "label"], spatial_size=ROI),
        ResizeWithPadOrCropd(["image", "label"], spatial_size=ROI),
        EnsureTyped(["image", "label"], track_meta=False),
        RandFlipd(["image", "label"], spatial_axis=i, prob=0.5) for i in range(3)
    ])

def val_tf():
    return Compose([
        LoadImaged(["image", "label"], image_only=False),
        EnsureChannelFirstd(["image", "label"]),
        Orientationd(["image", "label"], axcodes="RAS"),
        ScaleIntensityRanged("image", a_min=0, a_max=2500, b_min=0, b_max=1, clip=True),
        AsDiscreted("label"),
        SpatialPadd(["image", "label"], spatial_size=ROI),
        ResizeWithPadOrCropd(["image", "label"], spatial_size=ROI),
        EnsureTyped(["image", "label"]),
    ])

def load_fold(n):
    with open(DATASET_DIR / "manifests" / f"fold{n}.json") as f:
        m = json.load(f)
    return _load_cases(m, "train"), _load_cases(m, "validation")

print("Transforms ready.")

## 4. Training

In [ ]:
from monai.losses import DiceCELoss
from torch.amp import GradScaler, autocast
from tqdm.auto import tqdm

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def get_dice(pred, lbl, nc=3):
    dices = []
    for c in range(1, nc):
        p = (pred == c).float().flatten()
        t = (lbl == c).float().flatten()
        dices.append((2 * (p * t).sum() / (p.sum() + t.sum() + 1e-8)).item())
    return np.mean(dices)

def train_fold(fold_num):
    fold_dir = OUTPUT_DIR / f"fold{fold_num}"
    ckpt = fold_dir / "best_model.pth"

    if ckpt.exists():
        with open(fold_dir / "eval_metrics.json") as f:
            prev = json.load(f)
        print(f"  Fold {fold_num} done (DSC: {prev['best_dice']:.4f}) — skipping")
        return prev

    fold_dir.mkdir(parents=True, exist_ok=True)
    set_seed(SEED + fold_num)

    train_cases, val_cases = load_fold(fold_num)
    train_dl = DataLoader(CacheDataset(train_cases, train_tf(), cache_rate=1.0, num_workers=0),
                          BATCH_SIZE, shuffle=True, collate_fn=list_data_collate)
    val_dl   = DataLoader(CacheDataset(val_cases, val_tf(), cache_rate=1.0, num_workers=0), 1)

    model = HybridAttentionUNet3D(skip_mode=SKIP_MODE).to(device)
    weights = torch.tensor([0.167, 2.83, 3.07], device=device)
    loss_fn = DiceCELoss(include_background=False, to_onehot_y=True, softmax=True,
                         weight=weights, lambda_dice=1.5, lambda_ce=1.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.SequentialLR(optimizer, [
        torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=10),
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS - 10),
    ], [10])
    scaler = GradScaler()

    best_dice, patience, history = 0.0, 0, []

    for epoch in range(MAX_EPOCHS):
        model.train()
        tloss = 0
        for batch in train_dl:
            imgs, lbls = batch["image"].to(device), batch["label"].to(device)
            optimizer.zero_grad()
            with autocast("cuda"):
                loss = loss_fn(model(imgs), lbls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            tloss += loss.item()
        sched.step()
        tloss /= len(train_dl)

        model.eval()
        vdices = []
        with torch.no_grad():
            for batch in val_dl:
                imgs, lbls = batch["image"].to(device), batch["label"].to(device)
                out = model(imgs).argmax(dim=1, keepdim=True)
                vdices.append(get_dice(out, lbls))
        vdice = np.mean(vdices)
        lr = optimizer.param_groups[0]["lr"]
        history.append({"epoch": epoch+1, "train_loss": tloss, "val_dice": vdice, "lr": lr})

        if vdice > best_dice:
            best_dice, patience = vdice, 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "dice": best_dice}, ckpt)
            if (epoch+1) % 5 == 0 or epoch == 0:
                print(f"    Epoch {epoch+1:3d} | Loss: {tloss:.4f} | DSC: {vdice:.4f} | LR: {lr:.6f} [BEST]")
        else:
            patience += 1
            if (epoch+1) % 25 == 0:
                print(f"    Epoch {epoch+1:3d} | Loss: {tloss:.4f} | DSC: {vdice:.4f} | Pat: {patience}/{EARLY_STOP}")
        if patience >= EARLY_STOP:
            print(f"    Early stopping at epoch {epoch+1}")
            break

    pd.DataFrame(history).to_csv(fold_dir / "training_history.csv", index=False)
    result = {"fold": fold_num, "variant": VARIANT, "skip_mode": SKIP_MODE,
              "best_dice": best_dice, "epochs": len(history)}
    with open(fold_dir / "eval_metrics.json", "w") as f:
        json.dump(result, f, indent=2)
    print(f"    Done: DSC = {best_dice:.4f}")
    return result

print("Training functions ready.")

## 5. Run

In [ ]:
print(f"Running {VARIANT} (skip_mode={SKIP_MODE}) across {len(FOLDS)} folds\n")

all_results = []
for fold_num in FOLDS:
    print(f"Fold {fold_num}/{len(FOLDS)}")
    try:
        all_results.append(train_fold(fold_num))
    except Exception as e:
        print(f"  ERROR: {e}")

if all_results:
    dices = [r["best_dice"] for r in all_results]
    summary = {"variant": VARIANT, "skip_mode": SKIP_MODE,
               "folds_done": len(all_results),
               "mean_dice": float(np.mean(dices)), "std_dice": float(np.std(dices)),
               "fold_results": all_results}
    with open(OUTPUT_DIR / "cv_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\nMean DSC: {summary['mean_dice']:.4f} \u00b1 {summary['std_dice']:.4f}")

## 6. Results

In [ ]:
if all_results:
    df = pd.DataFrame(all_results)
    print(f"\n{'Fold':<8} {'Best DSC':<12} {'Epochs':<10}")
    print("-" * 30)
    for _, r in df.iterrows():
        print(f"{int(r['fold']):<8} {r['best_dice']:<12.4f} {int(r['epochs']):<10}")
    print("-" * 30)
    print(f"Mean: {df['best_dice'].mean():.4f} \u00b1 {df['best_dice'].std():.4f}")